In [21]:
import random
import numpy as np
from typing import List

# The Knapsack Problem

Given a set of items with weights and values, determine the maximum value you can carry within a strict weight limit.

In [11]:
def knapsack_2d(weights: list[int], values: list[int], capacity: int) -> int:
    """
    Standard 2D Dynamic Programming solution.
    Time Complexity: O(n * W)
    Space Complexity: O(n * W)
    """
    n = len(values)
    # Initialize an (n+1) x (capacity+1) matrix with zeros
    dp = [[0 for _ in range(capacity + 1)] for _ in range(n + 1)]

    # Build the DP table in bottom-up manner
    for i in range(1, n + 1):
        for w in range(1, capacity + 1):
            # If the current item fits in the knapsack
            if weights[i-1] <= w:
                # Max of (leave it, take it)
                dp[i][w] = max(dp[i-1][w], 
                               dp[i-1][w - weights[i-1]] + values[i-1])
            else:
                # Item is too heavy, we must leave it
                dp[i][w] = dp[i-1][w]

    return dp[n][capacity]

In [14]:
def knapsack_1d(weights: list[int], values: list[int], capacity: int) -> int:
    """
    Space-optimized 1D Dynamic Programming solution.
    Time Complexity: O(n * W)
    Space Complexity: O(W)
    """
    n = len(values)
    # We only need one array of size (capacity + 1)
    dp = [0] * (capacity + 1)

    for i in range(n):
        # Traverse the capacity backwards! 
        # We stop at weights[i] because below that, the item won't fit anyway.
        for w in range(capacity, weights[i] - 1, -1):
            dp[w] = max(dp[w], dp[w - weights[i]] + values[i])

    return dp[capacity]

In [18]:
item_weights = [3, 7, 12, 5, 6]
item_values = [76, 70, 80, 15, 10]
max_capacity = 30

max_value_2d = knapsack_2d(item_weights, item_values, max_capacity)
max_value_1d = knapsack_1d(item_weights, item_values, max_capacity)

print(f"Max Value (2D approach): {max_value_2d}")
print(f"Max Value (1D approach): {max_value_1d}")

Max Value (2D approach): 241
Max Value (1D approach): 241


# Longest Common Subsequence (LCS)

Finding the longest sequence that appears in the same relative order in two different strings.

In [19]:
def find_lcs(x: str, y: str) -> str:
    """
    Computes the Longest Common Subsequence (LCS) of two strings.
    
    Args:
        x (str): The first string.
        y (str): The second string.
        
    Returns:
        str: The longest common subsequence.
    """
    m, n = len(x), len(y)
    
    # Initialize an (m+1) x (n+1) matrix with zeros
    # dp[i][j] will store the length of the LCS for x[0..i-1] and y[0..j-1]
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # Step 1: Build the DP table bottom-up
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if x[i - 1] == y[j - 1]:
                # Characters match: extend the subsequence
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                # Characters differ: take the maximum of excluding either character
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    # Step 2: Traceback to construct the actual string
    # Start from the bottom-right corner of the matrix
    i, j = m, n
    lcs_chars = []
    
    while i > 0 and j > 0:
        if x[i - 1] == y[j - 1]:
            # If characters match, it is part of the LCS
            lcs_chars.append(x[i - 1])
            i -= 1
            j -= 1
        elif dp[i - 1][j] > dp[i][j - 1]:
            # Move in the direction of the larger value (Up)
            i -= 1
        else:
            # Move in the direction of the larger value (Left)
            j -= 1
            
    # The traceback collects characters in reverse order
    return "".join(reversed(lcs_chars))

In [20]:
X = "AGGTAB"
Y = "GXTXAYB"

lcs_string = find_lcs(X, Y)
print(f"String X: {X}")
print(f"String Y: {Y}")
print(f"LCS: {lcs_string}")
print(f"Length: {len(lcs_string)}")

String X: AGGTAB
String Y: GXTXAYB
LCS: GTAB
Length: 4


# Grid Walking

You are given an $m \times n$ grid filled with non-negative numbers representing costs. You start at the top-left $(0, 0)$ and want to reach the bottom-right $(m-1, n-1)$, moving only **down** or **right**. Your goal is to minimize the sum of the numbers along your path.

In [23]:
def min_path_sum(grid: List[List[int]]) -> int:
    """
    Calculates the minimum path sum from the top-left to the bottom-right of a grid.
    Optimized to use O(n) space complexity by maintaining a 1D state array.
    
    Args:
        grid: A 2D list of non-negative integers representing traversal costs.
        
    Returns:
        The minimum sum of the numbers along the optimal path.
    """
    if not grid or not grid[0]:
        return 0
        
    m, n = len(grid), len(grid[0])
    
    # Initialize a 1D DP array to store the current row's optimal path sums.
    dp = [0] * n
    dp[0] = grid[0][0]
    
    # Base Case: Initialize the first row. 
    # These cells can only be reached by moving straight right.
    for j in range(1, n):
        dp[j] = dp[j-1] + grid[0][j]
        
    # Process the remaining rows top-down.
    for i in range(1, m):
        # Base Case for the current row: The first column cell can only be 
        # reached by moving straight down from the cell directly above it.
        dp[0] += grid[i][0]
        
        # Traverse the columns left-to-right
        for j in range(1, n):
            # dp[j] evaluates to the optimal sum from the row above (i-1).
            # dp[j-1] evaluates to the newly computed optimal sum from the left (j-1).
            dp[j] = grid[i][j] + min(dp[j], dp[j-1])
            
    # The final element contains the minimum path sum to the bottom-right corner.
    return dp[-1]

In [24]:
cost_grid = [
    [1, 3, 1],
    [1, 5, 1],
    [4, 2, 1]
]

optimal_cost = min_path_sum(cost_grid)
print(f"The minimum path sum is: {optimal_cost}") # Expected Output: 7 (1 -> 3 -> 1 -> 1 -> 1)

The minimum path sum is: 7
